# 12.1 就地排序（sort）與新物件排序（sorted）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_12-1_in_place_sort_vs_sorted.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**先備知識**：已掌握 Chapter 8 串列基礎操作、記憶體參照概念與 Chapter 11 自訂函數回傳值。

---

### 學習導覽：從混亂走向有序——排序兩大神器的記憶體哲學

歡迎踏入演算法核心的殿堂——**排序（Sorting）與搜尋（Searching）**！在現實生活與競賽世界中，絕大多數的高效演算法（如二分搜尋、區間重疊判斷、活動貪婪排程、雙指標掃描）都建立在一個關鍵前提上：「**數據必須是有序的**」。

在 Python 語言中，要將雜亂無章的資料由小到大排好，我們有兩個最常用的超級法寶：
1. 串列內建方法：`list.sort()`
2. 內建通用函數：`sorted(iterable)`

表面上看，這兩個工具都能把串列排得整整齊齊，但如果分不清它們在「記憶體底層」的本質差異，就會寫出初學者最常犯的世紀大悲劇：`numbers = numbers.sort()`，導致整個串列直接蒸發成 `None`！

在本單元中，我們將透過 6 個平緩的微階梯，徹底搞懂兩者的記憶體行為與適用場景：
1. **12.1.1 排序核心概念**：什麼是排序？升序（Ascending）預設行為與由小到大排列。
2. **12.1.2 原地修改方法 `list.sort()`**：直接修改原串列，回傳值為 `None` 的本質。
3. **12.1.3 內建函數 `sorted()`**：保留原物件，產生全新獨立排序串列。
4. **12.1.4 記憶體耗損與使用時機對照**：原地修改 $O(1)$ 額外空間 vs 複製新串列 $O(N)$ 空間。
5. **12.1.5 非串列容器的排序**：以 `sorted()` 排序字串（回傳字元串列）、集合與元組。
6. **12.1.6 穩定排序（Stable Sort）特性**：數值相等元素之原始相對順序保持不變。

現在，讓我們啟動這趟排序演算法的探險之旅！

### 12.1.1 排序核心概念：什麼是排序？升序（Ascending）預設行為與由小到大排列

#### 1. 生活故事比喻：體育課排隊與圖書館還書
想像一下開學第一天的體育課，全班三十位同學剛走進操場時是四散在草地上聊天，雜亂無章。哨音一響，體育老師大喊一聲：「全體注意，矮的站前面、高的站後面，依照身高由矮到高排成一縱隊！」這就是生活中的**排序（Sorting）**。
排序的本質，就是將一群原本「雜亂無序」的元素，依照某種「嚴格的大小比較規則」，重新擺放到正確的位置上。在 Python 乃至於所有的程式語言中，如果沒有特別指定特殊規則，**預設的排序方向一律是「由小到大」**，在資訊科學中稱為**升序（Ascending Order）**。

#### 2. 底層運作機制：從離散混亂到單調遞增
在數學與電腦記憶體中，當我們給定一組數列 `[5, 2, 8, 1, 9]`，排序演算法會反覆檢視元素間的大小關係。
升序排序後的結果為 `[1, 2, 5, 8, 9]`。這滿足了數學上的單調非遞減特性：對於任意相鄰的兩個位置 $i$ 與 $i+1$，永遠滿足 `a[i] <= a[i+1]`。排序完成後，最小的元素永遠躺在第 0 個位置（`a[0]`），而最大的元素則必定出現在最後一個位置（`a[-1]`）。

#### 3. 初學者常見陷阱：以為所有型態都可以混合排序
初學同學最容易踩到的一個盲點，是把「不同型態」的資料塞在同一個串列中嘗試排序，例如 `[3, "apple", 1, "banana"]`。
在 Python 3 中，數字與字串之間是不能直接使用小於符號 `<` 進行比大小的。如果你嘗試對包含數字與字串混合的串列進行排序，直譯器會立刻憤怒地丟出 `TypeError: '<' not supported between instances of 'str' and 'int'` 錯誤！因此，排序的前提是元素之間必須具備「互相比較的共通標準」。

#### 4. APCS 實戰視野
在 APCS 競賽中，排序往往是許多困難演算法的「前置作業」。一旦數列由小到大排好，我們就能以 $O(1)$ 的極速取得極端值、以 $O(\log N)$ 進行二分搜尋，或以雙指標從頭尾相向夾擊。掌握升序的本質，是征服後續所有高階題目的第一塊基石。

In [ ]:
# 範例 12.1.1：升序排序預設行為感知
# 宣告一組未經排序的雜亂整數串列
scores = [78, 92, 65, 88, 54, 99, 70]
print("排序前的原始分數串列：", scores)

# 使用 sorted() 進行預設的由小到大（升序）排序
sorted_scores = sorted(scores)
print("升序排序後的成績串列：", sorted_scores)

# 驗證首尾元素：第一位必定是最低分，最後一位必定是最高分
print("全班最低分（索引 0）：", sorted_scores[0])
print("全班最高分（索引 -1）：", sorted_scores[-1])

In [ ]:
# 填空題 12.1.1：體驗由小到大升序排序
# 任務：將雜亂的身高資料由矮到高（升序）排序，並輸出排序結果與最高的身高數值。
heights = [165, 172, 158, 180, 169, 175]

# 請使用 sorted() 函數將 heights 進行升序排序
sorted_heights = ___(heights)

print("排序後的身高隊列：", sorted_heights)
# 請填入正確的索引，取出排序後隊列中「最高」的同學身高
print("隊列中最高的身高為：", sorted_heights[___])

In [ ]:
# ==========================================
# [4] Code 練習題 12.1.1
# 任務說明：
# 請撰寫一段程式，將給定的整數串列 raw_data 進行升序排序。
# 排序完成後，請依序印出：
# 1. 排序後的完整串列
# 2. 串列中的最小值與最大值之差（全距 = 最大值 - 最小值）
#
# 【公開測試資料 1】
# raw_data = [45, 12, 89, 34, 67]
# 預期輸出：
# 排序結果： [12, 34, 45, 67, 89]
# 全距： 77
#
# 【公開測試資料 2】
# raw_data = [5, 5, 5, 1]
# 預期輸出：
# 排序結果： [1, 5, 5, 5]
# 全距： 4
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
raw_data = [45, 12, 89, 34, 67]
res = sorted(raw_data)
print("排序結果：", res)
print("全距：", res[-1] - res[0])

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.1.1
# 任務說明：
# 某位同學在整理班上跑步測驗的秒數紀錄（包含浮點數）。
# 請宣告一個包含至少 5 筆浮點數的跑步成績串列，
# 使用升序排序後，印出前三名最快抵達終點的同學秒數（秒數越小代表越快）。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
times = [13.5, 12.1, 14.8, 11.9, 12.7, 13.0]
ordered_times = sorted(times)
print("跑步前三名最快秒數：", ordered_times[:3])

### 12.1.2 原地修改方法 `list.sort()`：直接修改原串列，回傳值為 `None` 的本質

#### 1. 生活故事比喻：在原本的房間重新擺設家具
想像你住在一間房間裡，房間內的書桌、椅子和書架原本擺得凌亂不堪。今天你捲起袖子花了一個下午，把「原本這間房間」裡面的所有家具重新推移就位，排列得井然有序。做完這件事後，你的房間還是原本那間房間（門牌號碼沒變），但是裡面的擺設已經被就地改寫了。
這就是串列的專屬方法 **`list.sort()`**！它直接在原串列所屬的記憶體空間內動手術，把裡面的元素重新調換位置，這種行為在電腦科學中被稱為**原地操作（In-place Modification）**。

#### 2. 底層運作機制：為什麼 `a.sort()` 回傳 `None`？
在 Python 的物件導向設計哲學中，有一條非常著名且嚴格的規範：**凡是直接在原物件身上就地修改的方法（如 `append()`, `extend()`, `reverse()`, `sort()`），其函數回傳值一律為 `None`！**
Python 設計之父 Guido van Rossum 刻意這樣設計，是為了向程式設計師發出明確的訊號：「提醒你，原物件已經被我修改了，所以我不需要另外回傳一個新的物件給你」。因此，執行 `a.sort()` 之後，直譯器默默做完了原處排序，回傳給呼叫者的只是一個空洞的 `None`。

#### 3. 初學者常見陷阱：世紀大悲劇 `a = a.sort()`
這是無數程式初學者在期中考與 APCS 考場上痛哭失聲的經典錯誤！許多初學者習慣了字串方法（如 `s = s.strip()`），於是在串列排序時寫下了：
```python
numbers = [3, 1, 2]
numbers = numbers.sort()  # 致命錯誤！
print(numbers)  # 輸出 None！原來的串列徹底蒸發！
```
當你寫 `numbers = numbers.sort()` 時，等號右側的 `numbers.sort()` 執行後回傳了 `None`，接著等號賦值將 `None` 硬生生覆蓋給了變數名牌 `numbers`！原本辛辛苦苦準備好的串列就這樣被垃圾回收機制銷毀，後續程式一旦呼叫 `numbers[0]` 就會直接崩潰引發 `TypeError: 'NoneType' object is not subscriptable`。

#### 4. APCS 實戰視野
在 APCS 實作題中，若題目的記憶體限制極為嚴苛，或者資料量高達幾十萬筆，使用 `a.sort()` 原地排序可以節省寶貴的記憶體空間。請務必牢記它的呼叫語法是**「單獨一行呼叫：`a.sort()`」**，千萬不要在它前面加上 `a =`！

In [ ]:
# 範例 12.1.2：list.sort() 的原地修改與回傳值驗證
nums = [40, 10, 30, 20]
print("排序前 nums 的記憶體位址（id）：", id(nums))
print("排序前的內容：", nums)

# 觀察 1：呼叫 .sort() 的回傳值是什麼？
return_val = nums.sort()
print("nums.sort() 執行的回傳值：", return_val)  # 必定為 None

# 觀察 2：原串列 nums 內部內容是否已被就地改變？
print("排序後 nums 的內容：", nums)

# 觀察 3：記憶體位址完全沒變，代表是同一個物件！
print("排序後 nums 的記憶體位址（id）：", id(nums))

In [ ]:
# 填空題 12.1.2：避免 a = a.sort() 陷阱
# 任務：正確使用 .sort() 原地排序串列，切勿覆蓋原變數！
cards = [7, 2, 9, 4, 1]

# 請「單獨一行」呼叫 cards 的原地排序方法，不要進行賦值
cards.___()

# 輸出原地排序後的撲克牌
print("整理後的撲克牌：", cards)

In [ ]:
# ==========================================
# [4] Code 練習題 12.1.2
# 任務說明：
# 給定一個包含負數與正數的整數串列 data。
# 請使用 .sort() 方法將 data 進行原地升序排序。
# 請注意：必須直接修改原串列，且不得產生 None！
# 最後請印出修改後的 data 內容與其第一個元素。
#
# 【公開測試資料 1】
# data = [15, -3, 0, 42, -9]
# 預期輸出：
# 原地排序後： [-9, -3, 0, 15, 42]
# 第一個元素： -9
#
# 【公開測試資料 2】
# data = [100, 50, 0]
# 預期輸出：
# 原地排序後： [0, 50, 100]
# 第一個元素： 0
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
data = [15, -3, 0, 42, -9]
data.sort()
print("原地排序後：", data)
print("第一個元素：", data[0])

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.1.2
# 任務說明：
# 請設計一個微型測試程式，專門用來警告粗心的隊友：
# 宣告一個串列 test_list = [5, 4, 3, 2, 1]
# 取得 test_list.sort() 的回傳值，
# 如果該回傳值是 None，請印出 "警告：.sort() 回傳 None，請勿寫出 a = a.sort()！"
# 否則印出原串列內容。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
test_list = [5, 4, 3, 2, 1]
ret = test_list.sort()
if ret is None:
    print("警告：.sort() 回傳 None，請勿寫出 a = a.sort()！")
    print("正確的原串列內容為：", test_list)

### 12.1.3 內建函數 `sorted(iterable)`：保留原物件，產生全新獨立排序串列

#### 1. 生活故事比喻：影印文件後在複本上重排順序
想像你手頭上有一份珍貴的歷史檔案原始手稿，裡面的各篇章節排列是依照成文年代記錄的。現在歷史老師請你繳交一份「依照文章字數由少到多排列」的作業，但同時要求你「絕對不能破壞或拆散原版手稿的原始順序」。
這時你會怎麼做？你一定會先將整份手稿放到影印機上「複印一份全新副本」，然後把原本珍貴的手稿放回防潮箱妥善保存，接著只在影印出來的紙張上重新打亂順序裝訂成冊。
這正是 Python 內建全能函數 **`sorted()`** 的運作方式！它**絕不觸碰、絕不修改**原物件，而是複製一份資料，排序完成後包裝成一個「全新的串列」回傳給你。

#### 2. 底層運作機制：物件身分證 `id()` 的分道揚鑣
當我們呼叫 `new_list = sorted(old_list)` 時，Python 底層在記憶體中進行了兩步操作：
1. 建立一個全新的 `list` 物件，把 `old_list` 中的元素參考複製一份過去。
2. 在這個全新的物件上執行排序，最後將新物件的記憶體參照回傳給等號左邊的 `new_list`。
如果我們使用身分證探測器 `id()` 分別檢驗 `old_list` 與 `new_list`，會發現兩者的記憶體位址截然不同（`id(old_list) != id(new_list)`）。原串列 `old_list` 裡面的元素毫髮無傷，依然維持著最初的混亂排列順序。

#### 3. 初學者常見陷阱：以為呼叫 `sorted(a)` 會直接改變 `a`
許多初學者在使用 `sorted()` 時，常會寫成：
```python
scores = [90, 70, 80]
sorted(scores)  # 以為 scores 就會排好
print(scores)   # 依然印出 [90, 70, 80]！
```
因為 `sorted()` 是純函數，它把排好序的全新名單「雙手奉上」回傳出來。如果沒有用變數接住它（例如 `ranked = sorted(scores)`），這個排序完的新串列就會因為沒有名牌參照而被直譯器當場丟棄！

#### 4. APCS 實戰視野
在 APCS 實作題目中，有時題目會要求我們：「輸出排序後的數列，但在後續計算中仍然需要用到資料最初輸入時的原始位置（例如需要原輸入的順序對應選手編號）」。這時候，`sorted()` 就是你的不二之選！保留原始數據的完整性，是避免連鎖邏輯錯誤的高級防禦機制。

In [ ]:
# 範例 12.1.3：sorted() 函數的無副作用與新物件生成
original = [50, 20, 40, 10, 30]
print("原始串列內容：", original)
print("原始串列記憶體位址（id）：", id(original))

# 呼叫 sorted()，必須用變數接收回傳的新串列
sorted_copy = sorted(original)

print("排序後的新串列內容：", sorted_copy)
print("排序後的新串列記憶體位址（id）：", id(sorted_copy))

# 檢驗原始串列：完全未被篡改！
print("檢驗原始串列，依然保持原樣：", original)
print("兩者是否為獨立物件？", id(original) != id(sorted_copy))

In [ ]:
# 填空題 12.1.3：用 sorted() 保留原始數據
# 任務：使用 sorted() 排序參賽者的成績，但必須同時保留原始登記順序。
log_order = [88, 95, 72, 60, 91]

# 請使用 sorted() 產出新串列，並賦值給 leaderboard
leaderboard = ___(log_order)

print("原始登記紀錄（不得被修改）：", log_order)
print("公佈之排行榜（已排序）：", leaderboard)

In [ ]:
# ==========================================
# [4] Code 練習題 12.1.3
# 任務說明：
# 某溫度計記錄了一天 24 小時中的 5 個採樣溫度 raw_temps。
# 請撰寫程式：
# 1. 產生一個新的排序串列 sorted_temps。
# 2. 印出原始採樣記錄（確認未被修改）。
# 3. 印出排序後的採樣記錄。
#
# 【公開測試資料 1】
# raw_temps = [28.5, 23.0, 31.2, 19.8, 25.4]
# 預期輸出：
# 原紀錄： [28.5, 23.0, 31.2, 19.8, 25.4]
# 排序後： [19.8, 23.0, 25.4, 28.5, 31.2]
#
# 【公開測試資料 2】
# raw_temps = [10.0, 5.0, 8.0]
# 預期輸出：
# 原紀錄： [10.0, 5.0, 8.0]
# 排序後： [5.0, 8.0, 10.0]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
raw_temps = [28.5, 23.0, 31.2, 19.8, 25.4]
sorted_temps = sorted(raw_temps)
print("原紀錄：", raw_temps)
print("排序後：", sorted_temps)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.1.3
# 任務說明：
# 某位學生做實驗時，想知道串列內元素的最大值與最小值分別出現在「原始輸入」中的哪一個索引位置。
# 請宣告一個至少包含 5 個不重複整數的串列 arr。
# 請利用 sorted(arr) 找出最小值與最大值，
# 接著利用原串列的 arr.index() 找出它們在原始串列中的位置並印出。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
arr = [34, 12, 89, 56, 7]
sorted_arr = sorted(arr)
min_val = sorted_arr[0]
max_val = sorted_arr[-1]
print(f"最小值 {min_val} 在原串列索引：{arr.index(min_val)}")
print(f"最大值 {max_val} 在原串列索引：{arr.index(max_val)}")

### 12.1.4 記憶體耗損與使用時機對照：原地修改 $O(1)$ 額外空間 vs 複製新串列 $O(N)$ 空間

#### 1. 生活故事比喻：辦公桌整理 vs 買一張新桌子
當你的辦公桌上堆滿了雜亂的文件時，你有兩種整理方式：
- **方案 A（原地整理）**：直接在原本這張桌子上把文件重新歸檔理順。你不需要再花錢買第二張桌子，也不需要額外的房間空間，額外佔用空間為 0。
- **方案 B（另購新桌）**：另外在隔壁房間再買一張全新一模一樣的桌子，把文件影印一份按照順序鋪在新桌上，舊桌子原封不動留著。雖然舊文件完整無缺，但你必須付出雙倍的桌面空間！
這就是 `list.sort()` 與 `sorted()` 在空間複雜度（Space Complexity）上的鮮明對照！

#### 2. 底層運作機制：$O(1)$ 與 $O(N)$ 的資源權衡
- **`list.sort()` 的空間複雜度為 $O(1)$ 輔助空間**：  
  Python 底層採用世界頂級的 **Timsort 演算法**（由 Tim Peters 於 2002 年設計）。當它在串列本身上面執行時，它是在原有的陣列記憶體區塊中交換指標，不需要開闢一整塊等長的新陣列，記憶體開銷極低。
- **`sorted()` 的空間複雜度為 $O(N)$ 輔助空間**：  
  如果原串列包含 $N = 1,000,000$（一百萬）個整數，呼叫 `sorted()` 會在記憶體中立刻額外開闢足以容納一百萬個元素的新串列空間。

| 比較項目 | `list.sort()` | `sorted(iterable)` |
| :--- | :--- | :--- |
| **呼叫方式** | `a.sort()`（物件專屬方法） | `sorted(a)`（通用內建函數） |
| **是否修改原串列** | 是（就地修改原物件） | 否（原物件完全不受影響） |
| **回傳值** | `None` | 全新的排序串列 |
| **額外記憶體空間** | $O(1)$（極省記憶體） | $O(N)$（需雙倍記憶體空間） |
| **適用對象** | 僅限串列 `list` | 任何可迭代物件（字串、元組、集合等） |

#### 3. 初學者常見陷阱：何時該用哪一個？
初學者常陷入選擇障礙。記憶心法非常單純：
1. **「原資料之後還要用嗎？」**：如果原順序絕對不能丟失（例如後續還要按原順序印出編號），**必選 `sorted()`**。
2. **「原資料不再需要，或數據量極大（幾十萬筆以上）？」**：為了極致效能與避免記憶體超標（MLE），**必選 `a.sort()`**。

#### 4. APCS 實戰視野
APCS 實作測驗對記憶體上限通常有嚴格限制（例如 256 MB 或 512 MB）。在處理大規模測資（例如 $N = 10^5$）時，養成「不需要保留原序列就使用 `a.sort()`」的好習慣，能確保你的程式絕不會因為記憶體超額而吞下 Memory Limit Exceeded (MLE) 判決！

In [ ]:
# 範例 12.1.4：list.sort() 與 sorted() 選擇策略演練
# 場景 A：大數據記憶體節約模式——原地排序
large_data = [i * 3 % 1000 for i in range(10)]
print("場景 A 原始數據：", large_data)
# 不需要保留原順序，直接原地排序，省去額外空間開銷
large_data.sort()
print("場景 A 原地排序結果：", large_data)

# 場景 B：雙軌保留模式——同時保留原始編號順序與名次排名
students_heights = [170, 162, 185, 175, 168]
# 學生的編號剛好是索引：0號=170, 1號=162 ...
ranked_heights = sorted(students_heights)
print("場景 B 原始座號身高表（供查詢）：", students_heights)
print("場景 B 由矮到高排隊表（供排隊）：", ranked_heights)

In [ ]:
# 填空題 12.1.4：根據場景選擇正確的排序工具
# 任務：
# 場景 1 需要節省記憶體並直接修改 nums1。
# 場景 2 需要保留 nums2 原樣，並將排序結果存入新變數 sorted_nums2。

nums1 = [9, 3, 7, 1]
# 請填寫適當的方法進行原地修改
nums1.___()
print("nums1 原地排序：", nums1)

nums2 = [8, 4, 6, 2]
# 請填寫適當的函數產出新串列
sorted_nums2 = ___(nums2)
print("nums2 保持原樣：", nums2)
print("sorted_nums2 全新排序：", sorted_nums2)

In [ ]:
# ==========================================
# [4] Code 練習題 12.1.4
# 任務說明：
# 某線上遊戲需要結算全服玩家戰力。
# 給定玩家戰力清單 power_list。
# 請先用 sorted() 產生戰力排行榜 leaderboard，
# 並輸出原始戰力清單的第一位玩家數值，以及排行榜最後一位玩家數值（戰力最高者）。
#
# 【公開測試資料 1】
# power_list = [2400, 1500, 3200, 1800, 2900]
# 預期輸出：
# 原始第一位玩家戰力： 2400
# 全服最高戰力： 3200
#
# 【公開測試資料 2】
# power_list = [999, 100, 500]
# 預期輸出：
# 原始第一位玩家戰力： 999
# 全服最高戰力： 999
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
power_list = [2400, 1500, 3200, 1800, 2900]
leaderboard = sorted(power_list)
print("原始第一位玩家戰力：", power_list[0])
print("全服最高戰力：", leaderboard[-1])

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.1.4
# 任務說明：
# 給定一個包含 8 個整數的串列 items。
# 請撰寫程式驗證記憶體節省行為：
# 1. 記錄執行 items.sort() 前後的 id(items)，驗證記憶體位址完全相同。
# 2. 另外宣告 items2，記錄 items2 與 sorted(items2) 的 id，驗證記憶體位址不同。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
items = [8, 2, 5, 1, 9, 3, 7, 4]
id_before = id(items)
items.sort()
id_after = id(items)
print("items 原地排序前後 id 是否相同：", id_before == id_after)

items2 = [10, 20, 30]
sorted_items2 = sorted(items2)
print("items2 與 sorted 副本 id 是否不同：", id(items2) != id(sorted_items2))

### 12.1.5 非串列容器的排序：以 `sorted()` 排序字串（回傳字元串列）、集合與元組

#### 1. 生活故事比喻：全能翻譯官與萬用收納盒
想像你手上有好幾種不同型式的雜物：有串在繩子上的珍珠項鍊（字串）、鎖在鐵盒子裡的硬幣（不可變的元組）、甚至是一袋隨意扔在一起的彈珠（無序的集合）。
今天你想要把裡面的東西通通由小到大排好。串列專屬的 `.sort()` 就像一個「只認識特定木箱的整理器」，只要遇到不是串列的容器（比如字串或元組），它就兩手一攤完全罷工。
但 `sorted()` 就像一位「精通萬物的萬能分檢官」，只要任何東西具備**可迭代（Iterable）**的特性（能一個個把裡面的東西掏出來），`sorted()` 就能把它們全部接納、依序排好，最後**一律包裝成整整齊齊的 Python 串列（`list`）**送還給你！

#### 2. 底層運作機制：跨型態通用性與統一輸出格式
- **字串排序**：字串雖然是不可變物件，但屬於可迭代序列。`sorted("python")` 會逐字取出字元，依照 ASCII 順序排好，回傳字元串列 `['h', 'n', 'o', 'p', 't', 'y']`。若想重新組合成字串，只需搭配 `''.join(...)`。
- **元組（Tuple）排序**：元組是不可變的，因此沒有 `tuple.sort()` 方法。但 `sorted((5, 1, 4))` 可以順利產出排序後的串列 `[1, 4, 5]`。
- **集合（Set）排序**：集合本身是無序且不重複的容器，沒有順序概念。但 `sorted({40, 10, 30})` 可以將其元素依序取出並排好，回傳有序串列 `[10, 30, 40]`！

#### 3. 初學者常見陷阱：對字串或元組呼叫 `.sort()` 引發 AttributeError
初學同學最容易發生的報錯之一：
```python
text = "banana"
text.sort()  # 崩潰！AttributeError: 'str' object has no attribute 'sort'

tp = (3, 1, 2)
tp.sort()    # 崩潰！AttributeError: 'tuple' object has no attribute 'sort'
```
請永遠記住鐵律：**只有串列 `list` 才有 `.sort()` 方法！** 面對字串、元組、集合或字典，唯一的排序選擇就是呼叫內建函數 **`sorted()`**。

#### 4. APCS 實戰視野
在字串處理題（例如檢驗兩個單字是否為「字母易位字 Anagram」，如 `"listen"` 與 `"silent"`），只要執行 `sorted(s1) == sorted(s2)`，一行就能秒殺判斷！學會用 `sorted()` 駕馭各種非串列容器，能大幅精簡競賽程式碼。

In [ ]:
# 範例 12.1.5：sorted() 跨容器排序示範
# 1. 字串排序：回傳排序後的字元串列，再以 join 組裝回字串
word = "algorithm"
sorted_chars = sorted(word)
print("字串經 sorted 後的型態與內容：", type(sorted_chars), sorted_chars)
rebuilt_word = "".join(sorted_chars)
print("使用 ''.join() 重新拼回純文字：", rebuilt_word)

# 2. 元組（Tuple）排序：元組不可變，但可透過 sorted 獲得有序串列
tup = (80, 20, 95, 45)
sorted_tup_list = sorted(tup)
print("元組排序後的串列：", sorted_tup_list)

# 3. 集合（Set）排序：集合無序去重，sorted 將其轉為有序串列
unique_nums = {5, 2, 8, 2, 5, 1}
sorted_set_list = sorted(unique_nums)
print("集合排序後的串列：", sorted_set_list)

In [ ]:
# 填空題 12.1.5：字母易位字（Anagram）檢驗神器
# 任務：判斷兩單字包含的字母是否完全相同（字母易位字）。
word1 = "listen"
word2 = "silent"

# 請填入正確的函數，分別對兩單字進行排序
is_anagram = ___(word1) == ___(word2)

print(f"'{word1}' 與 '{word2}' 是否為字母易位字？", is_anagram)

In [ ]:
# ==========================================
# [4] Code 練習題 12.1.5
# 任務說明：
# 給定一個包含重複數字的串列 raw_list。
# 請撰寫程式：
# 1. 利用集合 set() 先進行去重。
# 2. 利用 sorted() 將去重後的數字由小到大排序。
# 3. 印出去重且排序後的最終串列。
#
# 【公開測試資料 1】
# raw_list = [7, 2, 9, 2, 7, 1, 9, 3]
# 預期輸出：
# 去重並排序結果： [1, 2, 3, 7, 9]
#
# 【公開測試資料 2】
# raw_list = [10, 10, 10]
# 預期輸出：
# 去重並排序結果： [10]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
raw_list = [7, 2, 9, 2, 7, 1, 9, 3]
unique_sorted = sorted(set(raw_list))
print("去重並排序結果：", unique_sorted)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.1.5
# 任務說明：
# 某密碼驗證規則要求：使用者輸入的密碼字串，其內部字元依照 ASCII 順序重新排列後，
# 第一個字元必須是數字字元（即 '0' <= ch <= '9'）。
# 請撰寫一段程式碼，給定任意字串 password，
# 使用 sorted() 檢查重排後的第一個字元是否符合數字要求，
# 若符合印出 "合格"，否則印出 "不合格"。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
password = "Python3Security"
sorted_chars = sorted(password)
if "0" <= sorted_chars[0] <= "9":
    print("合格")
else:
    print("不合格")

### 12.1.6 穩定排序（Stable Sort）特性：數值相等元素之原始相對順序保持不變

#### 1. 生活故事比喻：同分同學的排隊先來後到
想像學校公布期末考成績，小明和小華兩位同學數學都考了整整 95 分。在原始登記名冊上，小明是在週一上午提早交卷登記的（排在前面），小華則是在週二下午交卷的（排在後面）。
今天教務處把成績依照分數由小到大排好公告。如果排序系統具備**「穩定排序（Stable Sort）」**的優秀品質，那麼當系統發現小明和小華分數完全相等時，系統會嚴格維持他們「在原本名冊上的先後順序」——也就是小明依然會排在小華前面，絕不會無緣無故把兩人的相對順序顛倒。

#### 2. 底層運作機制：Timsort 的穩定保證
Python 內建的 `sort()` 與 `sorted()` 所使用的 Timsort 演算法，是一套經過嚴謹數學證明的**保證穩定排序（Stable Sorting Algorithm）**演算法！
其定義為：**若輸入序列中有兩個元素 $A$ 與 $B$，其比對鍵值相等（$Key(A) == Key(B)$），且在原始序列中 $A$ 出現在 $B$ 之前，則排序之後 $A$ 依然保證出現在 $B$ 之前！**
這個看似不起眼的特性，是實現「多準則排序（先依 A 排序，相同再依 B 排序）」的核心靈魂！

#### 3. 初學者常見陷阱：以為所有排序演算法都具備穩定性
許多初學者在學校學習演算法時，常常誤以為「把數列排好就行了，穩定性無所謂」。
事實上，很多著名的傳統演算法（例如快速排序 Quick Sort、堆積排序 Heap Sort、選擇排序 Selection Sort）都是**不穩定排序（Unstable Sort）**！若在不穩定的排序演算法中排序，相同分數的資料前後順序會隨機亂跳。一旦後續有次要條件需要依賴前一步的排序結果，整個程式的邏輯就會徹底崩潰。

#### 4. APCS 實戰視野
APCS 觀念題常考「穩定排序」的定義與判斷。在實作題中，善用 Python 穩定的保證，我們可以進行多次排序：例如先針對「學號」排序一次，再針對「成績」排序一次；由於成績相同者會保持學號的原有順序，我們不費吹灰之力就達成了「成績相同時依學號升序」的複合排序需求！

In [ ]:
# 範例 12.1.6：驗證 Python 的穩定排序特性
# 串列中每個項目包含：(分數, 姓名)
# 注意：小明和小華的分數都是 90 分，但小明在原始名單中排在前方
records = [(85, "小強"), (90, "小明"), (75, "小英"), (90, "小華")]
print("原始登記順序：", records)

# 撰寫一個只提取「分數」作為排序依據的輔助小工具
# 雖然我們還沒正式教到自訂排序 key，但這裡先直觀展示相等鍵值的先後行為
def get_score(student):
    return student[0]

# 依分數由小到大排序
sorted_records = sorted(records, key=get_score)
print("排序後的成績單：", sorted_records)

# 驗證穩定性：檢驗兩位 90 分同學的相對先後
ninety_students = [name for score, name in sorted_records if score == 90]
print("90 分同學的先後名次：", ninety_students)  # 小明保證排在小華前面！

In [ ]:
# 填空題 12.1.6：穩定排序概念檢驗
# 任務：觀察帶有標籤的相同數值元素，驗證穩定排序前後相對順序不變。
data = [("A", 10), ("B", 5), ("C", 10), ("D", 5)]

def get_value(item):
    return item[1]

# 請使用 sorted 搭配 key=get_value 進行排序
sorted_data = ___(data, key=___)

# 預期：數值為 5 的兩項中，B 依然在 D 前面；數值為 10 的兩項中，A 依然在 C 前面
print("穩定排序驗證結果：", sorted_data)

In [ ]:
# ==========================================
# [4] Code 練習題 12.1.6
# 任務說明：
# 給定四位選手的測驗成績元組 (分數, 選手編號)。
# 請依分數排序（成績相同的選手必須嚴格維持原登記順序）。
# 印出排序後的選手清單。
#
# 【公開測試資料 1】
# athletes = [(100, "T1"), (80, "T2"), (100, "T3"), (80, "T4")]
# 預期輸出：
# 排序結果： [(80, 'T2'), (80, 'T4'), (100, 'T1'), (100, 'T3')]
#
# 【公開測試資料 2】
# athletes = [(50, "P1"), (50, "P2")]
# 預期輸出：
# 排序結果： [(50, 'P1'), (50, 'P2')]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def take_score(item):
    return item[0]

athletes = [(100, "T1"), (80, "T2"), (100, "T3"), (80, "T4")]
res = sorted(athletes, key=take_score)
print("排序結果：", res)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.1.6
# 任務說明：
# 請設計一個微型展示：
# 建立一個包含 6 筆資料的串列，每筆資料為 (商品類別, 商品名稱)。
# 其中至少有 3 件商品屬於同一類別。
# 請將這份資料依「商品類別」排序，並印出排序結果，
# 證明同一類別的商品，在排序後依然完全保持原本的先來後到次序。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def get_category(item):
    return item[0]

products = [
    ("文具", "原子筆"),
    ("飲料", "綠茶"),
    ("文具", "鉛筆盒"),
    ("零食", "洋芋片"),
    ("文具", "橡皮擦"),
    ("飲料", "紅茶")
]
sorted_products = sorted(products, key=get_category)
print("依類別穩定排序結果：")
for p in sorted_products:
    print(p)

### 學習總結與通關回顧

恭喜你順利通關 **12.1 就地排序（sort）與新物件排序（sorted）**！

在本單元中，你已經掌握了 Python 排序體系最核心的底層邏輯：
- **升序排列預設行為**：由小到大排列，`a[0]` 為最小值，`a[-1]` 為最大值。
- **原地操作 `list.sort()`**：
  - 專屬於 `list` 物件。
  - 直接在原物件記憶體空間中進行排列，$O(1)$ 額外空間極省記憶體。
  - 回傳值嚴格為 `None`，**絕對不能寫 `a = a.sort()`**！
- **純函數 `sorted(iterable)`**：
  - 適用於任何可迭代容器（串列、字串、元組、集合等）。
  - 保留原始物件不動，回傳全新的排序串列（需耗費 $O(N)$ 額外空間）。
  - 字串排序回傳字元串列，可搭配 `''.join()` 重新組裝。
- **穩定排序（Stable Sort）保證**：
  - 相同鍵值的元素在排序後依然保持原始相對次序，為後續多準則排序立下堅不可摧的根基。

---
**下一關預告**：想要由大到小降序排列該怎麼辦？文字數字字串 `'10'` 竟然比 `'2'` 還小？下一節 **12.2 逆序排序、字串字典序與數值轉型陷阱** 將為你揭開 ASCII 字典序的真相與考場防禦技巧！